# 01 — Dataset Familiarization (load raw)

City: **London** · Snapshot: **2025-09-14** · Source: [Inside Airbnb](https://insideairbnb.com/get-the-data/)

This notebook implements Phase 1 Step 5 of the plan: load the three primary raw files **without cleaning**.
No nulls are filled, no duplicates dropped, no columns renamed, no values coerced. Profiling and cleaning come in later steps.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 8)
pd.set_option("display.width", 140)

raw_dir = Path("..") / "data" / "raw" / "london"
print("raw_dir resolved to:", raw_dir.resolve())

raw_dir resolved to: C:\Users\Bavikaran\Desktop\New Folder\airbnb-assessment\data\raw\london


## Load listings (detailed)

In [2]:
listings = pd.read_csv(
    raw_dir / "listings.csv.gz",
    compression="gzip",
    low_memory=False,
)
print("listings shape:", listings.shape)
print("memory (MB):", round(listings.memory_usage(deep=True).sum() / 1_048_576, 1))

listings shape: (96871, 79)
memory (MB): 227.7


In [3]:
listings.head(3)

,id,listing_url,scrape_id,last_scraped,...,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,...,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,...,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,...,2,0,0,0.32


## Load calendar

Largest of the three — ~35M rows, ~1.2 GB uncompressed. Loading may take 30–60 s and a few GB of RAM.

In [4]:
calendar = pd.read_csv(
    raw_dir / "calendar.csv.gz",
    compression="gzip",
    low_memory=False,
)
print("calendar shape:", calendar.shape)
print("memory (MB):", round(calendar.memory_usage(deep=True).sum() / 1_048_576, 1))

calendar shape: (35357974, 7)
memory (MB): 2259.2


In [5]:
calendar.head(3)

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,1196288722069341420,2025-09-15,f,NaN,NaN,1,1125
1,1196288722069341420,2025-09-16,f,NaN,NaN,1,1125
2,1196288722069341420,2025-09-17,f,NaN,NaN,1,1125


## Load reviews (detailed)

In [6]:
reviews = pd.read_csv(
    raw_dir / "reviews.csv.gz",
    compression="gzip",
    low_memory=False,
)
print("reviews shape:", reviews.shape)
print("memory (MB):", round(reviews.memory_usage(deep=True).sum() / 1_048_576, 1))

reviews shape: (2097996, 6)
memory (MB): 631.5


In [7]:
reviews.head(3)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,13913,80770,2010-08-18,177109,Michael,My girlfriend and I hadn't known Alina before ...
1,13913,367568,2011-07-11,19835707,Mathias,Alina was a really good host. The flat is clea...
2,13913,529579,2011-09-13,1110304,Kristin,Alina is an amazing host. She made me feel rig...


## Sanity checks (no mutation)

Confirm row counts agree with the inventory, list dtypes pandas inferred, and surface obvious red flags for the profiling step.

In [8]:
summary = pd.DataFrame(
    [
        {"file": "listings.csv.gz", "rows": len(listings), "cols": listings.shape[1]},
        {"file": "calendar.csv.gz", "rows": len(calendar), "cols": calendar.shape[1]},
        {"file": "reviews.csv.gz",  "rows": len(reviews),  "cols": reviews.shape[1]},
    ]
)
summary

,file,rows,cols
0,listings.csv.gz,96871,79
1,calendar.csv.gz,35357974,7
2,reviews.csv.gz,2097996,6


In [9]:
print("--- listings dtypes (counts) ---")
print(listings.dtypes.value_counts())
print("\n--- calendar dtypes ---")
print(calendar.dtypes)
print("\n--- reviews dtypes ---")
print(reviews.dtypes)

--- listings dtypes (counts) ---
str        34
float64    25
int64      20
Name: count, dtype: int64

--- calendar dtypes ---
listing_id          int64
date                  str
available             str
price             float64
adjusted_price    float64
minimum_nights      int64
maximum_nights      int64
dtype: object

--- reviews dtypes ---
listing_id       int64
id               int64
date               str
reviewer_id      int64
reviewer_name      str
comments           str
dtype: object


In [10]:
# Quick look at suspicious type detections to confirm cleaning targets for Phase 2.
for col in ["price", "host_response_rate", "host_acceptance_rate",
            "host_is_superhost", "instant_bookable", "last_scraped",
            "first_review", "last_review"]:
    if col in listings.columns:
        sample = listings[col].dropna().head(3).tolist()
        print(f"{col:28} dtype={str(listings[col].dtype):8} sample={sample}")

price                        dtype=str      sample=['$70.00', '$149.00', '$411.00']
host_response_rate           dtype=str      sample=['100%', '88%', '100%']
host_acceptance_rate         dtype=str      sample=['96%', '50%', '88%']
host_is_superhost            dtype=str      sample=['t', 'f', 't']
instant_bookable             dtype=str      sample=['f', 'f', 'f']
last_scraped                 dtype=str      sample=['2025-09-16', '2025-09-16', '2025-09-16']
first_review                 dtype=str      sample=['2010-08-18', '2009-12-21', '2011-03-21']
last_review                  dtype=str      sample=['2025-08-21', '2025-04-05', '2024-02-19']


In [11]:
for col in ["available", "price", "adjusted_price", "date"]:
    if col in calendar.columns:
        sample = calendar[col].dropna().head(3).tolist()
        print(f"{col:18} dtype={str(calendar[col].dtype):8} sample={sample}")

available          dtype=str      sample=['f', 'f', 'f']
price              dtype=float64  sample=[]


adjusted_price     dtype=float64  sample=[]
date               dtype=str      sample=['2025-09-15', '2025-09-16', '2025-09-17']


## Notes carried forward

- Row counts match [`reports/dataset_inventory.csv`](../reports/dataset_inventory.csv): 96,871 listings, 35,357,974 calendar, 2,097,996 reviews.
- Listings `price`, `host_response_rate`, `host_acceptance_rate`, `host_is_superhost`, `instant_bookable`, and the listing date columns load as text (`str` in pandas 3). These are the type-repair targets for Phase 2.2.
- Calendar `price` / `adjusted_price` loaded as `float64` purely because every value is NaN — see the finding cell below.
- No mutations performed here. Profiling tables and the schema document are produced in Step 6.

## Critical finding: calendar price is 100% null

Out-of-notebook spot check confirmed across all 35,357,974 calendar rows:

```
available value counts:
  f  21,313,866   (60.3%)
  t  14,044,108   (39.7%)

price            null %: 100.00
adjusted_price   null %: 100.00
```

The `price` and `adjusted_price` columns exist in the calendar schema but every value is NaN in this London 2025-09-14 snapshot. The `available` column is fully populated.

**Consequences for the plan:**

- Availability and occupancy-proxy work in Phase 2.3 is unaffected.
- The original per-date revenue proxy from calendar prices cannot be computed from this snapshot.
- Fallbacks: (a) approximate revenue using listing-level `price` × unavailable-day count, (b) use the pre-computed `estimated_revenue_l365d` field in the detailed listings file (with the blocked-vs-booked caveat), or (c) drop revenue work and focus on availability and pricing snapshots.
- Add a new assumption (A-005) and a new limitation entry in Steps 11–12.